In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow import keras

In [ ]:
def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

In [ ]:
SAVE_DIR = "artifacts-behavorial"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv("robust_behavioral_data.csv", low_memory=False)

drop_cols = ["user_id", "resolution", "screen_width", "screen_height"]
X = df.drop(columns=drop_cols + ["label"])
y = df["label"]

In [ ]:
X.columns

In [ ]:
y

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

save_pickle(scaler, f"{SAVE_DIR}/scaler.pkl")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
lr_model = LogisticRegression(max_iter=500)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_pred)

In [ ]:
save_pickle(lr_model, f"{SAVE_DIR}/logistic_model.pkl")

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred)

In [ ]:
save_pickle(rf_model, f"{SAVE_DIR}/random_forest_model.pkl")

In [ ]:
input_shape = X_train.shape[1]

# Functional version of the same model
inputs = keras.Input(shape=(input_shape,), name="input")
x = keras.layers.Dense(32, activation="relu")(inputs)
x = keras.layers.Dense(16, activation="relu")(x)
outputs = keras.layers.Dense(1, activation="sigmoid", name="output")(x)

mlp_model = keras.Model(inputs=inputs, outputs=outputs, name="BehaviorMLP")


mlp_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC"])
mlp_model.fit(X_train, y_train, epochs=15, batch_size=32, validation_split=0.1, verbose=0)

mlp_pred = mlp_model.predict(X_test).flatten()
mlp_auc = roc_auc_score(y_test, mlp_pred)

In [ ]:
mlp_model.save(f"{SAVE_DIR}/behavior_mlp_model.keras")

In [ ]:
# ---------------- Compare Models ----------------
print(f"🔍 Logistic Regression AUC: {lr_auc:.4f}")
print(f"🌲 Random Forest AUC:       {rf_auc:.4f}")
print(f"🧠 MLP (Keras) AUC:         {mlp_auc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# ---------------- Plot ROC Curves ----------------
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_pred)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_pred)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_pred)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic (AUC={lr_auc:.3f})", linestyle="--")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={rf_auc:.3f})", linestyle="-.")
plt.plot(fpr_mlp, tpr_mlp, label=f"MLP (AUC={mlp_auc:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], color='gray', linestyle=':')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("📈 ROC Curves for Behavioral Models")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig("roc_curves.png")
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

def evaluate_model(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    print(f"\n📊 {name} Evaluation:")
    print(f"  AUC:       {auc:.4f}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")


In [ ]:
evaluate_model("Logistic Regression", y_test, lr_pred)
evaluate_model("Random Forest", y_test, rf_pred)
evaluate_model("MLP (Keras)", y_test, mlp_pred)


In [ ]:
best_model_name = "mlp"
best_auc = mlp_auc

if lr_auc > best_auc:
    best_model_name = "logistic"
    best_auc = lr_auc
if rf_auc > best_auc:
    best_model_name = "random_forest"
    best_auc = rf_auc

print(f"\n✅ Best model: {best_model_name} with AUC = {best_auc:.4f}")

# ---------------- Export Best Model to TFLite (only if MLP wins) ----------------
if best_model_name == "mlp":
    # Convert Sequential to Functional for safe TFLite export
    inputs = keras.Input(shape=(input_shape,))
    x = mlp_model(inputs)  # Reuse the trained model on new input
    export_model = keras.Model(inputs, x)

    converter = tf.lite.TFLiteConverter.from_keras_model(mlp_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

    # converter = tf.lite.TFLiteConverter.from_saved_model("artifacts-behavorial/behavior_mlp_model.keras")
    # converter.optimizations = [tf.lite.Optimize.DEFAULT]
    # converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
    
    tflite_model = converter.convert()

    with open("behavior_model.tflite", "wb") as f:
        f.write(tflite_model)

    print("📦 Exported MLP model to behavior_model.tflite")
else:
    print("ℹ️ Only MLP model is exportable to TFLite. No TFLite conversion done.")

print("✅ All models trained and evaluated.")



In [ ]:
export_model = keras.models.load_model(f"{SAVE_DIR}/behavior_mlp_model.keras")

# tflite model input, output shapes
print("\nTFLite Model Input Shape:", export_model.input_shape)
print("TFLite Model Output Shape:", export_model.output_shape)